# 弥漫性脑胶质瘤演化风险预测系统 (Glioma Evolutionary Risk Predictor)
**基于 TCGA 多组学数据的机器学习生存分析与特征提炼**

## 1. 项目科学背景与临床痛点
弥漫性低级别脑胶质瘤（LGG, WHO Grade II/III）是成年人中最常见的恶性原发性脑肿瘤之一。在临床治疗中，LGG 展现出极高的**肿瘤内异质性（Intratumoral Heterogeneity）**和**不可预测的演化轨迹**：
* 部分患者在手术后能维持长达十年的稳定期。
* 而另一部分患者则会在短短一两年内，迅速发生恶性克隆演化（Clonal Evolution），复发并升级为致命的胶质母细胞瘤（GBM, WHO Grade IV）。
传统的病理学分级和单一的基因检测（如 IDH 突变）已无法精准捕捉这种动态的演化风险。

## 2. 计算生物学解决方案与研究对标
本项目对标**香港科技大学（HKUST）王吉光教授课题组**在胶质瘤演化与计算神经肿瘤学领域的突破性研究（如 *CELLO* 模型与免疫微环境 *IME* 亚型分析）。
我们将利用人工智能与数据科学技术：
1. 接入 NIH 官方最高质控的 **TCGA-LGG PanCancer Atlas** 全面多组学队列。
2. 将临床生存期（OS/PFS）与超过 20,000 个基因的转录组表达水平（mRNA z-scores）进行高维对齐。
3. 利用底层向量化算力提取纯粹的数学特征张量，为后续构建 **XGBoost 机器学习生存预测模型**与 **SHAP 生物学可解释性分析**奠定黄金数据底座。

## Day 1：Pandas 高维矩阵读入与探索性数据分析 (EDA)
**学术复盘**：
真实世界的临床多组学数据往往存储为带注释的制表符分隔文件（TSV）。我们引入了数据科学核心引擎 `Pandas`，通过配置参数绕过文件头部的 metadata 注释（`#`），精准解析了 TCGA 队列。
通过 `df.info()` 和 `.shape` 属性的调用，我们完成了初步的**探索性数据分析（Exploratory Data Analysis, EDA）**。这避免了将庞大矩阵直接打印导致的内存溢出风险，并为后续锁定关键临床特征（如生存状态、年龄）确立了索引坐标。

In [ ]:
import pandas as pd
try:
    df = pd.read_csv("data_clinical_patient.txt", sep="\t", comment="#")
    print("数据读取成功！\n")
except FileNotFoundError:
    print(f"数据文件未找到，请检查文件路径：\n")
    df = None
if df is not None:
    print("---表格前五行数据---")
    print(df.head())
    print("\n---表格行列尺寸---")
    print(df.shape)
    print("\n---表格信息---")
    print(df.info())
    print("\n---提取表格前10行的前5列数据---")
    sub_matrix = df.iloc[0:10, 0:5]
    print(sub_matrix)

## Day 2：临床队列清洗准则与高危亚群截取
**学术复盘**：
医疗数据的最大痛点是**数据缺失（Missing Values）**。错误的填补会导致严重的“输入垃圾，输出垃圾（GIGO）”现象。我们执行了极其严格的医学统计学清洗准则：
1. **靶标签的零容忍**：生存期（OS_MONTHS）是监督学习的目标 $Y$，绝对禁止估算，缺失该项的患者必须执行硬性剔除（Drop）。
2. **特征的无偏插补**：对于少量缺失的年龄（AGE）特征，我们使用整个队列的中位数（Median）进行填补。相较于平均数，中位数能有效抵抗极值患者的偏差干扰。
3. **精准亚群切片**：运用高级布尔索引（Boolean Indexing），瞬间提纯了“高龄且已发生生存终点事件”的高危胶质瘤亚群。

In [ ]:
null_counts = df.isnull().sum()
print("各临床指标缺失值统计：")
print(null_counts[null_counts > 0]) 
df.dropna(subset=['OS_MONTHS', 'OS_STATUS'], inplace=True)
print(f"删除缺失生存标签的样本后，剩余的样本量为：{df.shape[0]}")
median_age = df['AGE'].median()
df['AGE'].fillna(median_age, inplace=True)
df['SEX'].fillna("Unknown", inplace=True)
high_risk_group = df[(df['AGE'] > 50) & (df['OS_STATUS'] == '1:DECEASED')]
print(f"筛选出高危（高龄且已死亡）样本数量为：{high_risk_group.shape[0]}")
df.to_csv("data_clinical_cleaned.csv", index=False)
print("清洗后的数据已保存到 data_clinical_cleaned.csv 文件中。")

## Day 3：TCGA PanCancer 黄金数据接入与集成测试
**学术复盘**：
为剥离批次效应（Batch Effect）等测序噪音，我们最终锚定了 NIH 官方最高质控标准的 **TCGA-LGG PanCancer Atlas** 数据集。
在提取多组学表格时，我探测到了全行业多组学对齐的核心障碍：**主键颗粒度异构**。即 Patient 表采用 12 位患者条码（如 `TCGA-DU-5872`），而表达量与病理表挂载于 15 位的生物样本条码（如 `TCGA-DU-5872-01`）。本次集成测试为次日的降维合并锚定了技术路线。


In [ ]:
df_patient = pd.read_csv('data_clinical_patient.txt', sep='\t', comment='#')
print(f"患者加载成功,尺寸为: {df_patient.shape}, 主键列示例: {df_patient['PATIENT_ID'].head(3).tolist()}")
df_sample = pd.read_csv('data_clinical_sample.txt', sep='\t', comment='#')
print(f"样本加载成功,尺寸为: {df_sample.shape}, 是否包含肿瘤级别分级列: { 'GRADE' in df_sample.columns }")
df_mrna = pd.read_csv('data_mrna_seq_v2_rsem_zscores_ref_all_samples.txt', sep='\t', comment='#')
print(f" mRNA 表达谱加载成功,尺寸为: {df_mrna.shape},包含的基因数量：{df_mrna.shape[0]}")

## Day 4：多组学主键降维对齐与表达谱高维转置
**学术复盘**：
本环节是构建 AI 特征库最核心的数据工程挑战，包含两大关键动作：
1. **特征矩阵翻转**：机器学习的规范输入要求 $\mathbf{X} \in \mathbb{R}^{n \times p}$ （行 $n$ 为患者，列 $p$ 为特征）。原始表达谱处于反置状态。我在剔除无效的 Entrez 数字编码后执行矩阵转置（`.T`），将 20,000 个基因符号无缝重组为自变量集合。
2. **防错对齐与双重内连接**：我创新性地通过 `.index.astype(str).str[:12]` 提取 12 位主键，彻底绕过了转置后的 Pandas 命名陷阱。随后通过 `how='inner'` 的双重融合运算，自动剔除了临床标签或分子测序残缺的“废弃样本”，产出了 100% 对齐的**终极超级多组学黄金表（Multi-omics Super Table）**。

In [ ]:
import numpy as np
df_patient = pd.read_csv('data_clinical_patient.txt', sep='\t', comment='#')
df_sample = pd.read_csv('data_clinical_sample.txt', sep='\t', comment='#')
df_mrna = pd.read_csv('data_mrna_seq_v2_rsem_zscores_ref_all_samples.txt', sep='\t', comment='#')
df_sample['PATIENT_ID'] = df_sample['SAMPLE_ID'].str[:12]
print("样本表与患者表主键ID已完成12位对齐")
df_mrna_clean = df_mrna.dropna(subset=['Hugo_Symbol'])
df_mrna_clean = df_mrna_clean.set_index('Hugo_Symbol')
if 'Entrez_Gene_Id' in df_mrna_clean.columns:
    df_mrna_clean = df_mrna_clean.drop(columns=['Entrez_Gene_Id'])
df_mrna_transposed = df_mrna_clean.T
df_mrna_transposed['PATIENT_ID'] = df_mrna_transposed.index.astype(str).str[:12]
df_mrna_transposed = df_mrna_transposed.reset_index(drop=True)
print(f"mRNA表达谱已完成转置，最新维度:样本{df_mrna_transposed.shape[0]} x 基因特征{df_mrna_transposed.shape[1]}")
df_clinical = pd.merge(df_sample, df_patient, on='PATIENT_ID', how='inner')
print(f"临床数据与样本数据已合并完成,样本数: {df_clinical.shape[0]},")
df_super = pd.merge(df_clinical, df_mrna_transposed, on='PATIENT_ID', how='inner')
print(f"最终整合数据集已完成,维度: {df_super.shape[0]}患者 x {df_super.shape[1]}特征")
required_col = ['PATIENT_ID', 'AGE','SEX','OS_STATUS','OS_MONTHS','GRADE']
for col in required_col:
    if col in df_super.columns:
        print(f"关键列 {col} 在最终数据集中存在")
if 'IDH1' in df_super.columns and 'TP53' in df_super.columns:
    print("基因表达数据已成功整合到最终数据集中")
df_super.to_csv('data_multiomics_merged.csv', index=False)

## Day 5：元数据剥离防数据泄露与 NumPy 算力提取
**学术复盘**：
在建立 XGBoost 等树模型前，必须对“答案”进行严格屏蔽（防止 **Data Leakage 数据泄露**）。
在超级大表中，包含了属于结果变量（Y）的生存状态和时间。我们通过基于列位置的硬切片 `df.iloc[:, 6:]` 将元数据剥离。最后，通过调用底层的 `.to_numpy()` 指令，将臃肿的 Pandas 数据框剥去了所有的字符表头与行号，淬炼成了**纯粹的高速浮点运算张量（NumPy Float64 Array）**，为下一周的 AI 算法训练扫清了最后的底层内存障碍。

In [ ]:
# 读入拼接好的超级多组学合并表
df_super = pd.read_csv("data_multiomics_merged.csv")

# 剥离临床标签（防数据泄露）：假设前 6 列是临床属性，从第 7 列开始全都是基因表达丰度
df_genes_only = df_super.iloc[:, 6:]

# 将 Pandas DataFrame 转化为同质的纯数值 NumPy 矩阵 X (供算法引擎食用)
X = df_genes_only.to_numpy()

print("--- 机器学习特征矩阵 X 提取质检 ---")
print(f"X 的底层数据类型 (dtype): {X.dtype} (应为 float64 数值)")
print(f"X 的矩阵尺寸 (Shape): {X.shape} (患者数 x 基因特征数)")